In [ ]:
# If needed, uncomment this once:
# !python -m pip uninstall -y opencv-python opencv-python-headless opencv-contrib-python opencv-contrib-python-headless mediapipe
# !python -m pip install --no-cache-dir "numpy==1.26.4" "protobuf==3.20.3" "opencv-contrib-python-headless==4.8.1.78"
# !python -m pip install --no-deps --no-cache-dir "mediapipe==0.10.9" tqdm pillow
!apt-get update && apt-get install -y ffmpeg

In [ ]:
import numpy
import cv2
import mediapipe as mp

print("numpy:", numpy.__version__)
print("cv2:", cv2.__version__)
print("mediapipe:", mp.__version__)

In [3]:
# ============================================================
# VIDEO -> FRAME EXTRACTION + BLUR/HAND TRIAGE (FIXED)
# Runs from /code/
# Saves outputs to /project/project/...
# ============================================================

import cv2
import csv
import shutil
import subprocess
from pathlib import Path
from collections import Counter
from tqdm import tqdm
import mediapipe as mp

# -----------------------------
# CONFIG (ABSOLUTE PATHS)
# -----------------------------
VIDEO_DIR = Path("/project/project/videos").resolve()

RAW_FRAMES_DIR = Path("/project/project/extracted_frames_raw").resolve()
SORTED_DIR = Path("/project/project/extracted_frames_sorted").resolve()

KEEP_DIR = SORTED_DIR / "keep"
REVIEW_DIR = SORTED_DIR / "review"
REJECT_DIR = SORTED_DIR / "reject"

FPS = 2
IMAGE_EXT = "png"
BLUR_THRESHOLD = 90.0
MIN_FACE_DETECTION_SCORE = 0.4
MIN_HAND_DETECTION_SCORE = 0.4

MIN_TOTAL_HAND_LANDMARKS = 12
GOOD_HAND_LANDMARKS = 16

CLEAR_OLD_OUTPUTS = False
FORCE_REEXTRACT = True

VALID_VIDEO_EXTS = {".mp4", ".mov", ".m4v", ".avi", ".mkv", ".webm"}

WRITE_CSV = True
CSV_PATH = Path("/project/project/frame_triage_results.csv")

# -----------------------------
# PREP FOLDERS
# -----------------------------
if CLEAR_OLD_OUTPUTS:
    for d in [RAW_FRAMES_DIR, SORTED_DIR]:
        if d.exists():
            shutil.rmtree(d)

for d in [RAW_FRAMES_DIR, KEEP_DIR, REVIEW_DIR, REJECT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("VIDEO_DIR:", VIDEO_DIR)
print("RAW_FRAMES_DIR:", RAW_FRAMES_DIR)
print("SORTED_DIR:", SORTED_DIR)

# -----------------------------
# HELPERS
# -----------------------------
def extract_frames_from_video(video_path: Path):
    video_name = video_path.stem
    out_dir = RAW_FRAMES_DIR / video_name

    if FORCE_REEXTRACT and out_dir.exists():
        shutil.rmtree(out_dir)

    out_dir.mkdir(parents=True, exist_ok=True)

    existing = list(out_dir.glob(f"*.{IMAGE_EXT}"))
    if existing and not FORCE_REEXTRACT:
        print(f"Skipping {video_path.name} (already extracted: {len(existing)} frames)")
        return

    output_pattern = str(out_dir / f"frame_%06d.{IMAGE_EXT}")

    cmd = [
        "ffmpeg",
        "-hide_banner",
        "-loglevel", "error",
        "-y",
        "-i", str(video_path),
        "-vf", f"fps={FPS}",
        output_pattern
    ]

    print(f"Extracting: {video_path.name}")
    subprocess.run(cmd, check=True)

    written = list(out_dir.glob(f"*.{IMAGE_EXT}"))
    if not written:
        raise RuntimeError(f"ffmpeg finished but no frames were written for {video_path.name} -> {out_dir}")

    print(f"Extracted {len(written)} frames -> {out_dir}")

def laplacian_variance(image_bgr):
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    return cv2.Laplacian(gray, cv2.CV_64F).var()

def count_visible_hand_landmarks(hand_landmarks):
    return sum(
        1 for lm in hand_landmarks.landmark
        if 0 <= lm.x <= 1 and 0 <= lm.y <= 1
    )

def analyze_frame(image_path: Path, hands_model, face_model):
    image_bgr = cv2.imread(str(image_path))
    if image_bgr is None:
        return {"ok": False}

    blur_score = laplacian_variance(image_bgr)

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    face_result = face_model.process(image_rgb)
    hands_result = hands_model.process(image_rgb)

    face_count = len(face_result.detections) if face_result.detections else 0

    hand_count = 0
    visible = []

    if hands_result.multi_hand_landmarks:
        hand_count = len(hands_result.multi_hand_landmarks)
        for h in hands_result.multi_hand_landmarks:
            visible.append(count_visible_hand_landmarks(h))

    return {
        "ok": True,
        "blur_score": blur_score,
        "face_count": face_count,
        "hand_count": hand_count,
        "total_visible_hand_landmarks": sum(visible)
    }

def classify_frame(stats):
    if not stats["ok"]:
        return "reject", "unreadable"

    if stats["blur_score"] < BLUR_THRESHOLD:
        return "reject", "blurry"

    if stats["face_count"] == 0:
        return "review", "no_face"

    if stats["hand_count"] == 0:
        return "keep", "face_only"

    if stats["total_visible_hand_landmarks"] >= GOOD_HAND_LANDMARKS:
        return "keep", "good_hands"

    if stats["total_visible_hand_landmarks"] >= MIN_TOTAL_HAND_LANDMARKS:
        return "review", "partial_hands"

    return "reject", "poor_hands"

def copy_frame(src: Path, cls: str, reason: str):
    dst_dir = SORTED_DIR / cls / src.parent.name
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst = dst_dir / f"{src.stem}__{reason}{src.suffix}"
    shutil.copy2(src, dst)
    return dst

# -----------------------------
# FIND VIDEOS
# -----------------------------
if not VIDEO_DIR.exists():
    raise FileNotFoundError(f"VIDEO_DIR does not exist: {VIDEO_DIR}")

video_files = sorted([
    f for f in VIDEO_DIR.iterdir()
    if f.is_file() and f.suffix.lower() in VALID_VIDEO_EXTS
])

if not video_files:
    print("Directory contents:")
    for f in VIDEO_DIR.iterdir():
        print(" -", f.name)
    raise FileNotFoundError("No videos found")

print("\nFound videos:")
for v in video_files:
    print(" -", v.name)

# -----------------------------
# EXTRACT FRAMES
# -----------------------------
for v in video_files:
    extract_frames_from_video(v)

# -----------------------------
# DEBUG RAW FRAME COUNTS
# -----------------------------
print("\nRaw frame folders:")
for d in sorted(RAW_FRAMES_DIR.glob("*")):
    if d.is_dir():
        count = len(list(d.glob(f"*.{IMAGE_EXT}")))
        print(f" - {d.name}: {count} frames")

# -----------------------------
# GATHER FRAMES
# -----------------------------
frames = sorted(RAW_FRAMES_DIR.rglob(f"*.{IMAGE_EXT}"))

print(f"\nTotal frames: {len(frames)}")

if not frames:
    raise FileNotFoundError(f"No frames extracted under {RAW_FRAMES_DIR}")

# -----------------------------
# ANALYZE + SORT
# -----------------------------
results = []

mp_hands = mp.solutions.hands
mp_face = mp.solutions.face_detection

with mp_hands.Hands(
        static_image_mode=True,
        max_num_hands=2,
        min_detection_confidence=MIN_HAND_DETECTION_SCORE
    ) as hands_model, mp_face.FaceDetection(
        model_selection=1,
        min_detection_confidence=MIN_FACE_DETECTION_SCORE
    ) as face_model:

    for f in tqdm(frames):
        stats = analyze_frame(f, hands_model, face_model)
        cls, reason = classify_frame(stats)
        dst = copy_frame(f, cls, reason)

        results.append({
            "src": str(f),
            "dst": str(dst),
            "class": cls,
            "reason": reason,
            "blur_score": stats.get("blur_score"),
            "face_count": stats.get("face_count"),
            "hand_count": stats.get("hand_count"),
            "total_visible_hand_landmarks": stats.get("total_visible_hand_landmarks"),
            "ok": stats.get("ok"),
        })

# -----------------------------
# OPTIONAL CSV REPORT
# -----------------------------
if WRITE_CSV:
    with open(CSV_PATH, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "src",
                "dst",
                "class",
                "reason",
                "blur_score",
                "face_count",
                "hand_count",
                "total_visible_hand_landmarks",
                "ok",
            ],
        )
        writer.writeheader()
        writer.writerows(results)

    print(f"\nCSV written: {CSV_PATH}")

# -----------------------------
# SUMMARY
# -----------------------------
counts = Counter(r["class"] for r in results)

print("\nSummary:")
for k in ["keep", "review", "reject"]:
    print(f"{k}: {counts.get(k, 0)}")

print("\nDone.")

VIDEO_DIR: /project/project/videos
RAW_FRAMES_DIR: /project/project/extracted_frames_raw
SORTED_DIR: /project/project/extracted_frames_sorted

Found videos:
 - IMG_0642.MOV
 - IMG_2643.MOV
 - IMG_4414.MOV
 - IMG_5364.MOV
 - IMG_5393.MOV
 - IMG_5839.MOV
 - IMG_6793.MOV
 - IMG_9088.MOV
 - IMG_9348.MOV
Extracting: IMG_0642.MOV
Extracted 46 frames -> /project/project/extracted_frames_raw/IMG_0642
Extracting: IMG_2643.MOV
Extracted 60 frames -> /project/project/extracted_frames_raw/IMG_2643
Extracting: IMG_4414.MOV
Extracted 33 frames -> /project/project/extracted_frames_raw/IMG_4414
Extracting: IMG_5364.MOV
Extracted 34 frames -> /project/project/extracted_frames_raw/IMG_5364
Extracting: IMG_5393.MOV
Extracted 14 frames -> /project/project/extracted_frames_raw/IMG_5393
Extracting: IMG_5839.MOV
Extracted 43 frames -> /project/project/extracted_frames_raw/IMG_5839
Extracting: IMG_6793.MOV
Extracted 31 frames -> /project/project/extracted_frames_raw/IMG_6793
Extracting: IMG_9088.MOV
Extracted

E0000 00:00:1775246436.313323    2766 gl_context.cc:408] INTERNAL: ; RET_CHECK failure (mediapipe/gpu/gl_context_egl.cc:303) successeglMakeCurrent() returned error 0x3008;  (entering GL context)
E0000 00:00:1775246436.313401    2766 gl_context.cc:408] INTERNAL: ; RET_CHECK failure (mediapipe/gpu/gl_context_egl.cc:303) successeglMakeCurrent() returned error 0x3008;  (entering GL context)
E0000 00:00:1775246436.313416    2766 gl_context.cc:408] INTERNAL: ; RET_CHECK failure (mediapipe/gpu/gl_context_egl.cc:303) successeglMakeCurrent() returned error 0x3008;  (entering GL context)
E0000 00:00:1775246436.322412    2766 gl_context.cc:408] INTERNAL: ; RET_CHECK failure (mediapipe/gpu/gl_context_egl.cc:303) successeglMakeCurrent() returned error 0x3008;  (entering GL context)
E0000 00:00:1775246436.322443    2766 gl_context.cc:408] INTERNAL: ; RET_CHECK failure (mediapipe/gpu/gl_context_egl.cc:303) successeglMakeCurrent() returned error 0x3008;  (entering GL context)
INFO: Created TensorFlow 

Extracted 31 frames -> /project/project/extracted_frames_raw/IMG_9348

Raw frame folders:
 - IMG_0642: 46 frames
 - IMG_2643: 60 frames
 - IMG_4414: 33 frames
 - IMG_5364: 34 frames
 - IMG_5393: 14 frames
 - IMG_5839: 43 frames
 - IMG_6793: 31 frames
 - IMG_9088: 31 frames
 - IMG_9348: 31 frames

Total frames: 323


  0%|          | 0/323 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/google/protobuf/symbol_database.py:78: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
100%|██████████| 323/323 [00:27<00:00, 11.81it/s]


CSV written: /project/project/frame_triage_results.csv

Summary:
keep: 13
review: 27
reject: 283

Done.
